# Extension: Hybrid BM25 + Sentence-BERT Retrieval and Cross-Encoder Reranking

This notebook extends the legal retrieval system to a full **two-stage pipeline** and evaluates it end-to-end on legal data.

Concretely, the code does the following:

1. **Builds a retrieval corpus and gold spans**  
   - Loads legal documents and gold answer snippets from the benchmark data.  
   - Chunks documents into passages that form the retrieval corpus.  
   - Maintains a mapping from each query to its gold answer passages.

2. **Implements and evaluates first-stage retrievers**  
   - **BM25 (lexical retrieval)** – captures exact term matches and rare legal keywords.  
   - **Sentence-BERT (dense retrieval)** – captures paraphrases and semantic similarity between queries and passages.  
   - **Hybrid BM25 + Sentence-BERT** – combines normalized BM25 and dense scores (and optionally RRF) to exploit complementary lexical and semantic signals.  
   - Optionally compares against a TF–IDF baseline.

3. **Constructs a candidate pool for reranking**  
   - For each query, uses the hybrid retriever to select the top-N most relevant passages.  
   - This candidate pool defines an **upper bound** on what any reranker can achieve, since gold passages must first appear among these candidates.

4. **Trains a legal-domain cross-encoder reranker (using LePard)**  
   - Uses the LePard citation dataset to create positive and negative `(quote, destination_context)` pairs.  
   - Fine-tunes a cross-encoder to predict a relevance score for each pair, learning legal citation-style relevance.

5. **Reranks candidates with the cross-encoder**  
   - For each query, scores all `(query, candidate_passage)` pairs with the cross-encoder.  
   - Produces a new ranking by either:
     - Using cross-encoder scores alone, or  
     - **Fusing** cross-encoder scores with hybrid scores:
       \[
       s_{\text{fused}} = \alpha \cdot s_{\text{CE}} + (1 - \alpha) \cdot s_{\text{hybrid}}
       \]
   - This focuses on improving the ordering **within** the candidate pool.

6. **Evaluates all methods under the same retrieval metrics**  
   - **Exact Match** (strict top-1 snippet match)  
   - **Span-level F1** (token overlap between predicted and gold spans)  
   - **Recall@K** (fraction of queries where a gold passage appears in the top-K)  
   - **nDCG@K** (ranking quality in the top-K)  

Overall, this milestone compares:
- classical lexical retrieval (TF–IDF, BM25),
- dense retrieval (Sentence-BERT),
- hybrid BM25 + Sentence-BERT, and
- cross-encoder reranking (both baseline and fine-tuned),

and analyzes how much each stage improves retrieval quality on legal text and how close the system gets to the candidate upper bound.


## Setup and Installation

In [1]:
# Installing required packages
!pip install rank-bm25 nltk scikit-learn numpy pandas sentence-transformers torch -q

In [2]:
import os
import re
import nltk
import json
import numpy as np
from rank_bm25 import BM25Okapi
from collections import defaultdict
from typing import List, Dict, Tuple, Optional
from sentence_transformers import SentenceTransformer

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Downloading stopwords
try:
    stop_words = set(stopwords.words('english'))
except:
    import ssl
    try:
        _create_unverified_https_context = ssl._create_unverified_context
    except AttributeError:
        pass
    else:
        ssl._create_default_https_context = _create_unverified_https_context
    nltk.download('stopwords', quiet=True)
    stop_words = set(stopwords.words('english'))

## Utilities Functions

In [3]:
def preprocess_text(text: str, lower: bool = True) -> str:
    """Preprocess text for retrieval."""
    if lower:
        text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def tokenize(text: str, remove_stopwords: bool = False) -> List[str]:
    """Tokenize text."""
    text = preprocess_text(text, lower=True)
    tokens = word_tokenize(text)
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words and t.isalnum()]
    else:
        tokens = [t for t in tokens if t.isalnum()]
    return tokens

def load_test_benchmark(benchmark_path: str) -> List[dict]:
    """Load test benchmark JSON file."""
    with open(benchmark_path, 'r') as f:
        data = json.load(f)
    return data.get('tests', data)

def load_corpus_file(corpus_path: str) -> str:
    """Load a corpus text file."""
    with open(corpus_path, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

def get_passage_from_span(text: str, span: List[int]) -> str:
    """Extract passage from text using character span."""
    start, end = span
    return text[start:end]

def chunk_text(text: str, chunk_size: int = 500) -> List[str]:
    """Split text into overlapping chunks."""
    words = text.split()
    chunks = []
    step = max(1, chunk_size // 2)
    for i in range(0, len(words), step):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
        if i + chunk_size >= len(words):
            break
    return chunks if chunks else [text]

def prepare_corpus_from_benchmark(
    benchmark_path: str,
    corpus_dir: str,
    chunk_size: int = 500) -> Tuple[List[str], Dict[str, List[str]]]:
    """
    Prepare corpus by chunking documents and mapping queries to gold passages.

    Returns:
        corpus_passages: List of all passages in corpus
        query_to_gold: Mapping from query to list of gold answer passages
    """
    tests = load_test_benchmark(benchmark_path)
    corpus_passages = []
    query_to_gold = {}
    processed_files = set()

    for test in tests:
        query = test['query']
        gold_answers = []

        for snippet in test.get('snippets', []):
            file_path = snippet['file_path']
            span = snippet['span']

            full_path = os.path.join(corpus_dir, file_path)
            if os.path.exists(full_path):
                doc_text = load_corpus_file(full_path)
                gold_passage = get_passage_from_span(doc_text, span)
                gold_answers.append(gold_passage)

                if full_path not in processed_files:
                    chunks = chunk_text(doc_text, chunk_size)
                    corpus_passages.extend(chunks)
                    processed_files.add(full_path)

        if gold_answers:
            query_to_gold[query] = gold_answers

    return corpus_passages, query_to_gold

### Preparing the LegalBench-RAG Corpus

The notebook first converts the LegalBench-RAG ContractNLI benchmark into a retrieval corpus.

For each test instance:

- The benchmark specifies a `file_path` pointing to a contract file and a character `span` for the gold snippet.
- The full document is loaded from the corpus directory.
- The gold snippet is extracted from the document using the span and stored as a **gold answer passage**.
- The full document is then split into overlapping chunks of around 500 words, which are added to a **shared retrieval corpus**.

A mapping is maintained from each query to its list of gold answer passages. After preprocessing, the setup typically looks like:

- A few hundred corpus passages
- Close to a thousand test queries with at least one gold snippet

This forms a realistic retrieval scenario: multiple queries, each with one or more gold answers embedded in a moderately sized corpus of contract passages.


## Hybrid Retriever Implementation

In [4]:
class SentenceBERTEncoder:
    """Encodes passages/queries with Sentence-BERT."""

    def __init__(
        self,
        model_name: str = "sentence-transformers/all-mpnet-base-v2",
        batch_size: int = 32,
        device: Optional[str] = None,
        normalize_embeddings: bool = True,
    ):
        self.model_name = model_name
        self.batch_size = batch_size
        self.normalize_embeddings = normalize_embeddings
        self.device = device
        self.model = SentenceTransformer(model_name, device=device)
        self.corpus_texts: List[str] = []
        self.corpus_embeddings: Optional[np.ndarray] = None

    def index(self, passages: List[str]):
        """Encode and store corpus passages."""
        if not passages:
            raise ValueError("Cannot index empty passage list.")
        self.corpus_texts = passages
        embeddings = self.model.encode(
            passages,
            batch_size=self.batch_size,
            convert_to_numpy=True,
            show_progress_bar=True,
            normalize_embeddings=self.normalize_embeddings,
        )
        if not self.normalize_embeddings:
            norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
            norms[norms == 0] = 1.0
            embeddings = embeddings / norms
        self.corpus_embeddings = embeddings.astype(np.float32)

    def score(self, query: str) -> np.ndarray:
        """Score query against all corpus passages."""
        if self.corpus_embeddings is None:
            raise ValueError("Index before scoring.")
        query_emb = self.model.encode(
            [query],
            batch_size=1,
            convert_to_numpy=True,
            show_progress_bar=False,
            normalize_embeddings=self.normalize_embeddings,
        )[0]
        if not self.normalize_embeddings:
            denom = np.linalg.norm(query_emb) + 1e-12
            query_emb = query_emb / denom
        scores = np.dot(self.corpus_embeddings, query_emb)
        return scores

### Hybrid BM25 + Sentence-BERT Retriever

To obtain a strong first-stage retriever, the notebook implements a **hybrid retrieval model** that combines:

1. **BM25 (lexical retrieval)**  
   - Operates on tokenized text.
   - Rewards exact matches of terms, especially rare legal vocabulary and citation patterns.

2. **Sentence-BERT dense retrieval**  
   - Encodes passages and queries into fixed-size embeddings using `all-mpnet-base-v2`.
   - Scores passages via cosine similarity between query and passage embeddings.
   - Captures paraphrastic and semantic similarity beyond exact word overlap.

For each query:

- BM25 returns a relevance score for every passage based on lexical match.
- Sentence-BERT returns a similarity score for every passage based on embeddings.
- The two score vectors are **normalized to [0, 1]** and then linearly combined:
  \[
  s_{\text{hybrid}} = w_{\text{BM25}} \cdot \tilde{s}_{\text{BM25}} + w_{\text{dense}} \cdot \tilde{s}_{\text{dense}}
  \]
  with tuned weights (e.g., **0.55 for BM25** and **0.45 for Sentence-BERT**).

This hybrid scoring scheme exploits the **complementarity** between strict lexical matching and more flexible semantic similarity, which is particularly valuable in legal text where both specific citations and paraphrased obligations appear.


In [5]:
class HybridRetriever:
    """
    Combines BM25 and Sentence-BERT scores via fusion.
    Supports two fusion methods: weighted combination and Reciprocal Rank Fusion (RRF).
    """

    def __init__(
        self,
        bm25_k1: float = 1.5,
        bm25_b: float = 0.75,
        model_name: str = "sentence-transformers/all-mpnet-base-v2",
        dense_batch_size: int = 32,
        fusion_method: str = "weighted",
        bm25_weight: float = 0.55,
        dense_weight: float = 0.45,
        rrf_k: int = 60,
        fusion_depth: int = 100,
        device: Optional[str] = None,
    ):
        self.bm25_k1 = bm25_k1
        self.bm25_b = bm25_b
        self.fusion_method = fusion_method
        self.bm25_weight = bm25_weight
        self.dense_weight = dense_weight
        self.rrf_k = rrf_k
        self.fusion_depth = fusion_depth

        self.bm25 = None
        self.encoder = SentenceBERTEncoder(
            model_name=model_name,
            batch_size=dense_batch_size,
            device=device,
            normalize_embeddings=True,
        )
        self.corpus_texts: List[str] = []
        self.tokenized_passages: List[List[str]] = []

    def index(self, passages: List[str]):
        """Index passages with both BM25 and Sentence-BERT."""
        self.corpus_texts = passages
        self.tokenized_passages = [
            tokenize(p, remove_stopwords=True) for p in passages
        ]
        self.bm25 = BM25Okapi(
            self.tokenized_passages, k1=self.bm25_k1, b=self.bm25_b
        )
        self.encoder.index(passages)

    def _normalize_scores(self, scores: np.ndarray) -> np.ndarray:
        """Normalize scores to [0, 1] range."""
        if scores.size == 0:
            return scores
        min_val = scores.min()
        max_val = scores.max()
        if max_val - min_val < 1e-9:
            return np.ones_like(scores)
        return (scores - min_val) / (max_val - min_val)

    def _fuse_rrf(
        self, bm25_scores: np.ndarray, dense_scores: np.ndarray
    ) -> Dict[int, float]:
        """Reciprocal Rank Fusion (RRF)."""
        rrf_scores: Dict[int, float] = defaultdict(float)

        bm25_rank = np.argsort(bm25_scores)[::-1][:self.fusion_depth]
        dense_rank = np.argsort(dense_scores)[::-1][:self.fusion_depth]

        for rank, idx in enumerate(bm25_rank):
            rrf_scores[int(idx)] += 1.0 / (self.rrf_k + rank + 1)

        for rank, idx in enumerate(dense_rank):
            rrf_scores[int(idx)] += 1.0 / (self.rrf_k + rank + 1)

        return rrf_scores

    def _fuse_weighted(
        self, bm25_scores: np.ndarray, dense_scores: np.ndarray
    ) -> Dict[int, float]:
        """Weighted combination of normalized scores."""
        bm25_norm = self._normalize_scores(bm25_scores)
        dense_norm = self._normalize_scores(dense_scores)
        combined = (
            self.bm25_weight * bm25_norm + self.dense_weight * dense_norm
        )
        return {int(idx): float(score) for idx, score in enumerate(combined)}

    def retrieve(self, query: str, k: int = 10) -> List[Tuple[str, float]]:
        """Retrieve top-k passages using hybrid fusion."""
        if self.bm25 is None or self.encoder.corpus_embeddings is None:
            raise ValueError("Retriever must be indexed before retrieval.")

        bm25_scores = self.bm25.get_scores(
            tokenize(query, remove_stopwords=True)
        )
        dense_scores = self.encoder.score(query)

        if self.fusion_method == "rrf":
            fused_scores = self._fuse_rrf(bm25_scores, dense_scores)
        else:
            fused_scores = self._fuse_weighted(bm25_scores, dense_scores)

        ranked = sorted(
            fused_scores.items(), key=lambda x: x[1], reverse=True
        )[:k]
        return [(self.corpus_texts[idx], score) for idx, score in ranked]

## Data Preparation

In [6]:
from google.colab import files
import zipfile
import io

# Uploading the ZIP file
uploaded = files.upload()

# Extracting the uploaded ZIP file
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(uploaded[filename]), 'r') as zip_ref:
            zip_ref.extractall('unzipped')
        print(f"Extracted {filename} to /content/unzipped")
    else:
        print(f"{filename} is not a ZIP file.")

Saving data_extracted.zip to data_extracted.zip
Extracted data_extracted.zip to /content/unzipped


In [7]:
BENCHMARK_PATH = '/content/unzipped/data_extracted/benchmarks/contractnli.json'
CORPUS_DIR = '/content/unzipped/data_extracted/corpus'
OUTPUT_DIR = '/content/output'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# Loading benchmark
print(f"Loading benchmark from {BENCHMARK_PATH}...")
tests = load_test_benchmark(BENCHMARK_PATH)
print(f"Loaded {len(tests)} test cases")

# Preparing corpus
print(f"Preparing corpus from {CORPUS_DIR}...")
corpus_passages, query_to_gold = prepare_corpus_from_benchmark(
    BENCHMARK_PATH,
    CORPUS_DIR,
    chunk_size=500
)
print(f"Corpus size: {len(corpus_passages)} passages")
print(f"Number of test queries: {len(query_to_gold)}")

Output directory: /content/output
Loading benchmark from /content/unzipped/data_extracted/benchmarks/contractnli.json...
Loaded 977 test cases
Preparing corpus from /content/unzipped/data_extracted/corpus...
Corpus size: 563 passages
Number of test queries: 977


## Running Hybrid Baseline

In [8]:
import nltk

# Ensuring NLTK tokenizers are available
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [9]:
print("Hybrid Baseline: BM25 + Sentence-BERT Fusion")

if len(corpus_passages) > 0 and len(tests) > 0:
    # Initializing hybrid retriever with weighted fusion
    # Tuned weights: 0.55 BM25, 0.45 Sentence-BERT
    hybrid_retriever = HybridRetriever(
        fusion_method="weighted",
        bm25_weight=0.55,
        dense_weight=0.45,
        fusion_depth=100,
        model_name="sentence-transformers/all-mpnet-base-v2",
        dense_batch_size=32,
        device=None,  # Auto-detect GPU if available
    )

    # Indexing corpus
    print(f"Indexing {len(corpus_passages)} passages with BM25 and Sentence-BERT...")
    hybrid_retriever.index(corpus_passages)

    # Generating predictions
    num_tests = len(tests) # Processing all queries
    print(f"Processing {num_tests} test queries...")
    hybrid_predictions = []
    for i, test in enumerate(tests[:num_tests]):
        if (i + 1) % 10 == 0:
            print(f"  Processed {i + 1}/{num_tests} queries...")
        query = test['query']
        results = hybrid_retriever.retrieve(query, k=10)
        retrieved_passages = [passage for passage, score in results]
        hybrid_predictions.append({
            'query': query,
            'retrieved_passages': retrieved_passages
        })

    # Saving predictions
    output_path = f'{OUTPUT_DIR}/hybrid_predictions.json'
    with open(output_path, 'w') as f:
        json.dump(hybrid_predictions, f, indent=2)
    print(f"Predictions saved to {output_path}")
else:
    print("Skipping hybrid baseline: corpus or tests not loaded")
    hybrid_predictions = []

Hybrid Baseline: BM25 + Sentence-BERT Fusion


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexing 563 passages with BM25 and Sentence-BERT...


Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Processing 977 test queries...
  Processed 10/977 queries...
  Processed 20/977 queries...
  Processed 30/977 queries...
  Processed 40/977 queries...
  Processed 50/977 queries...
  Processed 60/977 queries...
  Processed 70/977 queries...
  Processed 80/977 queries...
  Processed 90/977 queries...
  Processed 100/977 queries...
  Processed 110/977 queries...
  Processed 120/977 queries...
  Processed 130/977 queries...
  Processed 140/977 queries...
  Processed 150/977 queries...
  Processed 160/977 queries...
  Processed 170/977 queries...
  Processed 180/977 queries...
  Processed 190/977 queries...
  Processed 200/977 queries...
  Processed 210/977 queries...
  Processed 220/977 queries...
  Processed 230/977 queries...
  Processed 240/977 queries...
  Processed 250/977 queries...
  Processed 260/977 queries...
  Processed 270/977 queries...
  Processed 280/977 queries...
  Processed 290/977 queries...
  Processed 300/977 queries...
  Processed 310/977 queries...
  Processed 320/9

## Evaluation

### Evaluation Metrics

All retrieval systems in this notebook are evaluated using span- and ranking-based metrics tailored to text retrieval with gold snippets:

- **Exact Match**  
  Checks whether the *top-1* retrieved passage exactly matches any gold snippet after normalization. This is a very strict metric, and in legal corpora it is often close to zero because retrieved passages are larger chunks that only partially overlap the gold spans.

- **Span-level F1**  
  Computes token-level F1 between the top-1 retrieved passage and the best-matching gold passage:
  - Precision = fraction of retrieved tokens that are also in the gold snippet.
  - Recall = fraction of gold tokens that are also in the retrieved passage.
  - F1 is the harmonic mean of precision and recall.
  This metric captures **partial overlaps**, which are common in practice.

- **Recall@K**  
  For each query, measures the fraction of gold passages that appear within the top-K retrieved passages (based on overlap heuristics). The metric is then averaged across queries. It answers:
  > “In the top K passages, how often does the system retrieve at least one correct answer?”

- **nDCG@K (Normalized Discounted Cumulative Gain)**  
  Evaluates not just whether relevant passages are retrieved, but **how high** they appear in the ranking. Gains are discounted logarithmically by rank and normalized by the ideal ordering. This metric is sensitive to ranking quality within the top-K results.

Together, these metrics characterize both **hit quality** (Exact Match, span F1) and **ranking quality** (Recall@K, nDCG@K).


In [10]:
# Evaluation functions (from score.py)
def exact_match(predicted: str, gold: str) -> bool:
    """Check if predicted text exactly matches gold text."""
    return predicted.strip().lower() == gold.strip().lower()

def span_f1(predicted: str, gold: str) -> float:
    """Compute F1 score based on token overlap."""
    pred_tokens = set(tokenize(predicted, remove_stopwords=False))
    gold_tokens = set(tokenize(gold, remove_stopwords=False))

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    intersection = pred_tokens & gold_tokens
    precision = len(intersection) / len(pred_tokens) if len(pred_tokens) > 0 else 0.0
    recall = len(intersection) / len(gold_tokens) if len(gold_tokens) > 0 else 0.0

    if precision + recall == 0:
        return 0.0

    return 2 * (precision * recall) / (precision + recall)

def recall_at_k(retrieved_passages: List[str], gold_passages: List[str], k: int = 10) -> float:
    """Compute Recall@K: fraction of gold passages found in top K."""
    if len(gold_passages) == 0:
        return 1.0 if len(retrieved_passages) == 0 else 0.0

    top_k = retrieved_passages[:k]
    gold_normalized = [preprocess_text(g).strip() for g in gold_passages]
    retrieved_normalized = [preprocess_text(r).strip() for r in top_k]

    matches = 0
    for gold in gold_normalized:
        for ret in retrieved_normalized:
            if gold in ret or ret in gold or gold == ret:
                matches += 1
                break

    return matches / len(gold_passages)

def ndcg_at_k(retrieved_passages: List[str], gold_passages: List[str], k: int = 10) -> float:
    """Compute Normalized Discounted Cumulative Gain (nDCG) at K."""
    if len(gold_passages) == 0:
        return 1.0 if len(retrieved_passages) == 0 else 0.0

    top_k = retrieved_passages[:k]
    gold_normalized = [preprocess_text(g).strip() for g in gold_passages]
    retrieved_normalized = [preprocess_text(r).strip() for r in top_k]

    relevances = []
    for ret in retrieved_normalized:
        is_relevant = False
        for gold in gold_normalized:
            if gold in ret or ret in gold or gold == ret:
                is_relevant = True
                break
        relevances.append(1.0 if is_relevant else 0.0)

    dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(relevances))
    num_relevant = int(min(sum(relevances), k))
    idcg = sum(1.0 / np.log2(i + 2) for i in range(num_relevant))

    if idcg == 0:
        return 0.0

    return dcg / idcg

def evaluate_retrieval(
    predictions: List[Dict],
    gold_standard: List[Dict],
    k: int = 10
) -> Dict[str, float]:
    """Evaluate retrieval system performance."""
    if len(predictions) != len(gold_standard):
        raise ValueError(f"Mismatch: {len(predictions)} predictions vs {len(gold_standard)} gold examples")

    exact_matches = []
    span_f1_scores = []
    recall_at_k_scores = []
    ndcg_at_k_scores = []

    for pred, gold in zip(predictions, gold_standard):
        gold_answers = [snippet['answer'] for snippet in gold.get('snippets', [])]

        if len(gold_answers) == 0:
            continue

        retrieved = pred.get('retrieved_passages', [])

        if len(retrieved) == 0:
            exact_matches.append(0.0)
            span_f1_scores.append(0.0)
            recall_at_k_scores.append(0.0)
            ndcg_at_k_scores.append(0.0)
            continue

        top_pred = retrieved[0] if retrieved else ""
        em = any(exact_match(top_pred, gold_ans) for gold_ans in gold_answers)
        exact_matches.append(1.0 if em else 0.0)

        best_f1 = max([span_f1(top_pred, gold_ans) for gold_ans in gold_answers])
        span_f1_scores.append(best_f1)

        rec_k = recall_at_k(retrieved, gold_answers, k=k)
        recall_at_k_scores.append(rec_k)

        ndcg_k = ndcg_at_k(retrieved, gold_answers, k=k)
        ndcg_at_k_scores.append(ndcg_k)

    return {
        'exact_match': np.mean(exact_matches),
        'span_f1': np.mean(span_f1_scores),
        f'recall@{k}': np.mean(recall_at_k_scores),
        f'ndcg@{k}': np.mean(ndcg_at_k_scores),
        'num_examples': len(predictions)
    }

In [11]:
# Evaluating hybrid baseline predictions
if hybrid_predictions:
    hybrid_results = evaluate_retrieval(hybrid_predictions, tests[:len(hybrid_predictions)], k=10)
    print("=" * 60)
    print("HYBRID BASELINE RESULTS")
    print("=" * 60)
    for metric, value in hybrid_results.items():
        print(f"  {metric}: {value:.4f}")
    print("=" * 60)

    # Save results
    results_path = f'{OUTPUT_DIR}/hybrid_results.json'
    with open(results_path, 'w') as f:
        json.dump(hybrid_results, f, indent=2)
    print(f"Results saved to {results_path}")
else:
    print("No predictions to evaluate. Please run the hybrid baseline cell first.")

HYBRID BASELINE RESULTS
  exact_match: 0.0000
  span_f1: 0.2357
  recall@10: 0.5511
  ndcg@10: 0.4808
  num_examples: 977.0000
Results saved to /content/output/hybrid_results.json


### Hybrid Baseline on LegalBench-RAG

On the LegalBench-RAG ContractNLI benchmark, the hybrid BM25 + Sentence-BERT retriever produces results of the following form (example values):

- **Exact match ≈ 0.0**
- **Span F1 ≈ 0.23–0.24**
- **Recall@10 ≈ 0.55**
- **nDCG@10 ≈ 0.48**

Interpretation:

- Exact match is essentially zero because the evaluation compares **exact snippet strings**, while the retriever returns **larger document chunks** that only partially overlap the gold spans.
- Span F1 in the ~0.23–0.24 range indicates that the best retrieved passage typically overlaps a non-trivial portion of the gold snippet.
- Recall@10 around 0.55 means that in more than half of the queries, **at least one relevant passage** appears in the top-10 results.
- nDCG@10 around 0.48 indicates that relevant passages tend to appear reasonably high in the ranked list, though not consistently at the very top.

This hybrid retriever is subsequently used as the **candidate generator** for the cross-encoder reranker.


### Candidate Pool

The cross-encoder reranker does not score the entire corpus directly; instead, it operates over a **candidate pool** generated by the hybrid retriever.

For each query:

- The hybrid retriever returns the top-N passages.
- These candidates are stored as:
  ```python
  {
      "query": <query_text>,
      "candidate_passages": [p1, p2, ..., pN]
  }


In [12]:
import os
import tarfile
import pandas as pd
from sklearn.model_selection import train_test_split

LEPARD_TAR_PATH = "/content/training_top_100000_data.tar.gz"
LEPARD_EXTRACT_DIR = "/content/lepard"

os.makedirs(LEPARD_EXTRACT_DIR, exist_ok=True)

# Extract CSV from tar.gz
with tarfile.open(LEPARD_TAR_PATH, "r:gz") as tar:
    tar.extractall(path=LEPARD_EXTRACT_DIR)

# Find the CSV file inside the extracted folder
lepard_csv_files = [
    os.path.join(LEPARD_EXTRACT_DIR, f)
    for f in os.listdir(LEPARD_EXTRACT_DIR)
    if f.endswith(".csv")
]
assert len(lepard_csv_files) == 1, f"Expected 1 CSV, found: {lepard_csv_files}"
LEPARD_CSV_PATH = lepard_csv_files[0]
print("Using LePaRD CSV:", LEPARD_CSV_PATH)

/tmp/ipython-input-3497239056.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=LEPARD_EXTRACT_DIR)


Using LePaRD CSV: /content/lepard/top_100000_data.csv


In [13]:
# Load
lepard_df = pd.read_csv(LEPARD_CSV_PATH)

# Keep only rows with both quote and destination_context present
lepard_df = lepard_df.dropna(subset=["quote", "destination_context"]).reset_index(drop=True)
print("LePaRD rows after dropping NA:", len(lepard_df))

# Shuffle & split 80/20
lepard_df = lepard_df.sample(frac=1.0, random_state=42).reset_index(drop=True)

train_df, dev_df = train_test_split(
    lepard_df,
    test_size=0.2,
    random_state=42,
)

print("Train size:", len(train_df))
print("Dev size:", len(dev_df))

LePaRD rows after dropping NA: 100000
Train size: 80000
Dev size: 20000


In [14]:
from sentence_transformers import InputExample
import random

def build_pairwise_examples(df, max_pairs=None):
    """
    Create balanced positive/negative pairs:
      - pos: (quote, destination_context), label=1.0
      - neg: (quote, destination_context_from_other_row), label=0.0
    """
    df = df.reset_index(drop=True)
    quotes = df["quote"].tolist()
    contexts = df["destination_context"].tolist()

    # positives
    pos_pairs = list(zip(quotes, contexts))

    # negatives by cyclic shift of contexts (ensures different row)
    neg_contexts = contexts[1:] + contexts[:1]
    neg_pairs = list(zip(quotes, neg_contexts))

    examples = []
    for (q, c_pos), (_, c_neg) in zip(pos_pairs, neg_pairs):
        examples.append(InputExample(texts=[q, c_pos], label=1.0))
        examples.append(InputExample(texts=[q, c_neg], label=0.0))

    if max_pairs is not None and len(examples) > max_pairs:
        examples = random.sample(examples, max_pairs)

    return examples

# Limit to keep training lightweight
MAX_TRAIN_EXAMPLES = 80000  # pairs (pos+neg), so ~40k rows

train_examples = build_pairwise_examples(train_df, max_pairs=MAX_TRAIN_EXAMPLES)
dev_examples   = build_pairwise_examples(dev_df,   max_pairs=20000)

print("Train examples:", len(train_examples))
print("Dev examples:", len(dev_examples))


Train examples: 80000
Dev examples: 20000


In [16]:
from torch.utils.data import DataLoader
from sentence_transformers import CrossEncoder

BATCH_SIZE = 16
NUM_EPOCHS = 1
LEARNING_RATE = 2e-5
FINETUNED_CE_PATH = "crossencoder_lepard_finetuned"
BASE_CE_MODEL = "cross-encoder/ms-marco-MiniLM-L-12-v2"

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)

# Re-initialize a fresh copy of the base model for training
ce_for_training = CrossEncoder(BASE_CE_MODEL, num_labels=1, max_length=512)

warmup_steps = int(len(train_dataloader) * NUM_EPOCHS * 0.1)

ce_for_training.fit(
    train_dataloader=train_dataloader,
    epochs=NUM_EPOCHS,
    warmup_steps=warmup_steps,
    output_path=FINETUNED_CE_PATH,
    optimizer_params={"lr": LEARNING_RATE},
    use_amp=True,  # mixed precision if available
)

print("Saved fine-tuned cross-encoder to:", FINETUNED_CE_PATH)


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: snavya (snavya-university-of-pennsylvania) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,0.838600
1000,0.381100
1500,0.347900
2000,0.328800
2500,0.308100
3000,0.312800
3500,0.296000
4000,0.279400
4500,0.278700
5000,0.264100


Saved fine-tuned cross-encoder to: crossencoder_lepard_finetuned




### Set up configuration

In [17]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from sentence_transformers import InputExample
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator

# ----- CONFIG -----
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Base cross-encoder model
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Paths
OUTPUT_DIR = "/content/output"
CE_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "cross_encoder")
os.makedirs(CE_OUTPUT_DIR, exist_ok=True)

FINETUNED_CE_PATH = os.path.join(CE_OUTPUT_DIR, "lepard_finetuned_ce")

# LePard CSV path
LEPARD_CSV_PATH = "lepard/top_100000_data.csv"

# Training hyperparameters
NUM_EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1          # 10% warmup
SEED = 42

# Evaluation hyperparameters
TOP_K = 10  # for reranking & recall@k, ndcg@k


In [18]:
def build_hybrid_candidates(hybrid_predictions, top_M: int = 50):
    """
    Convert hybrid retrieval output into candidate lists
    for the cross-encoder reranker.

    Returns a list of dicts:
      [{"query": q, "candidate_passages": [p1, p2, ...]}, ...]
    """
    candidates = []
    for entry in hybrid_predictions:
        q = entry["query"]
        cands = entry.get("retrieved_passages", [])[:top_M]
        candidates.append({
            "query": q,
            "candidate_passages": cands
        })
    return candidates

# Build candidates (top 50 hybrid candidates per query, tunable)
hybrid_candidates = build_hybrid_candidates(hybrid_predictions, top_M=50)
print(f"Built {len(hybrid_candidates)} candidate sets.")


Built 977 candidate sets.


In [19]:
# Build evaluator for dev set (binary classification on LePard)
dev_evaluator = CEBinaryClassificationEvaluator.from_input_examples(
    dev_examples,
    name="lepard-dev"
)


In [20]:
total_training_steps = len(train_dataloader) * NUM_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_training_steps)
print("Total training steps:", total_training_steps)
print("Warmup steps:", warmup_steps)


Total training steps: 15000
Warmup steps: 1500


In [21]:
# Fresh instance for training
ce_for_training = CrossEncoder(
    CROSS_ENCODER_MODEL,
    num_labels=1,          # binary classification (0/1)
    max_length=512,
    device=DEVICE
)

# Train with evaluation on dev after each epoch
ce_for_training.fit(
    train_dataloader=train_dataloader,
    epochs=NUM_EPOCHS,
    warmup_steps=warmup_steps,
    output_path=FINETUNED_CE_PATH,   # will save best model here
    evaluator=dev_evaluator,
    evaluation_steps=len(train_dataloader),  # evaluate once per epoch
    optimizer_params={"lr": LEARNING_RATE},
    use_amp=True,  # mixed precision if available
)

# Just in case, also explicitly save final model state
ce_for_training.save(FINETUNED_CE_PATH)
print(f"Fine-tuned model saved to {FINETUNED_CE_PATH}")


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1006 > 512). Running this sequence through the model will result in indexing errors


Step,Training Loss,Validation Loss,Lepard-dev Accuracy,Lepard-dev Accuracy Threshold,Lepard-dev F1,Lepard-dev F1 Threshold,Lepard-dev Precision,Lepard-dev Recall,Lepard-dev Average Precision
5000,0.308400,No log,0.870450,0.215342,0.873600,-0.290357,0.843161,0.906318,0.954043
10000,0.241400,No log,0.894600,0.271888,0.895300,0.165065,0.893529,0.897079,0.965414
15000,0.205600,No log,0.901750,0.025805,0.902797,-0.684601,0.882906,0.923604,0.968998


Fine-tuned model saved to /content/output/cross_encoder/lepard_finetuned_ce


### Upper Bound

This design introduces a hard ceiling on what any reranker can achieve:

If a gold passage never appears among the hybrid-retrieved candidates, no reranking strategy—no matter how accurate—can retrieve it in the top-K.

By computing a “candidate upper bound” Recall@10, the notebook measures the maximum possible Recall@10 achievable given this candidate set.

On LegalBench-RAG, this upper bound is approximately:

Candidate upper bound Recall@10 ≈ 0.5855

Any reranker that operates only on these candidates therefore cannot exceed this ceiling. The best fused hybrid + cross-encoder configuration approaches this bound, indicating that the remaining headroom is primarily limited by first-stage retrieval rather than reranking quality.

In [22]:
def compute_candidate_upper_bound(candidates, gold_tests, k: int = 10):
    """
    For each query, check if any of the candidate_passages
    contains the gold snippet (or vice versa).
    This is an optimistic upper bound on recall@k for the reranker.
    """
    from copy import deepcopy

    def norm(x):
        return preprocess_text(x).strip()

    recalls = []
    for entry, gold in zip(candidates, gold_tests):
        cand_passages = entry["candidate_passages"][:k]
        gold_answers = [snippet["answer"] for snippet in gold.get("snippets", [])]
        if not gold_answers:
            continue

        cand_norm = [norm(p) for p in cand_passages]
        gold_norm = [norm(g) for g in gold_answers]

        hit = 0
        for g in gold_norm:
            for c in cand_norm:
                if g in c or c in g or g == c:
                    hit = 1
                    break
            if hit == 1:
                break
        recalls.append(hit)

    return float(np.mean(recalls)) if recalls else 0.0

candidate_upper_bound = compute_candidate_upper_bound(
    candidates=hybrid_candidates,
    gold_tests=tests,
    k=TOP_K
)
print(f"Candidate upper bound recall@{TOP_K}: {candidate_upper_bound:.4f}")


Candidate upper bound recall@10: 0.5855


## Cross Encoder Reranker + Finetuning

In [23]:
import numpy as np

CANDIDATE_K = 50   # number of candidates per query for reranker
TOP_K = 10         # final top-k for evaluation

def build_hybrid_candidates_with_scores(
    tests,
    hybrid_retriever,
    candidate_k: int = CANDIDATE_K
):
    """
    For each query, get top-k hybrid candidates and keep both texts and scores.
    Returns a list of dicts:
      {
        "query": str,
        "candidate_passages": [p1, p2, ...],
        "candidate_scores":   [s1, s2, ...]   # hybrid scores
      }
    """
    candidates = []
    num_tests = len(tests)
    print(f"Building hybrid candidates for {num_tests} queries (k={candidate_k})...")

    for i, test in enumerate(tests):
        if (i + 1) % 50 == 0:
            print(f"  Processed {i + 1}/{num_tests} queries...")

        query = test["query"]
        results = hybrid_retriever.retrieve(query, k=candidate_k)
        passages = [p for p, s in results]
        scores   = [s for p, s in results]

        candidates.append({
            "query": query,
            "candidate_passages": passages,
            "candidate_scores": scores
        })

    return candidates

# Build candidates once and reuse
hybrid_candidates = build_hybrid_candidates_with_scores(
    tests=tests,
    hybrid_retriever=hybrid_retriever,
    candidate_k=CANDIDATE_K
)


Building hybrid candidates for 977 queries (k=50)...
  Processed 50/977 queries...
  Processed 100/977 queries...
  Processed 150/977 queries...
  Processed 200/977 queries...
  Processed 250/977 queries...
  Processed 300/977 queries...
  Processed 350/977 queries...
  Processed 400/977 queries...
  Processed 450/977 queries...
  Processed 500/977 queries...
  Processed 550/977 queries...
  Processed 600/977 queries...
  Processed 650/977 queries...
  Processed 700/977 queries...
  Processed 750/977 queries...
  Processed 800/977 queries...
  Processed 850/977 queries...
  Processed 900/977 queries...
  Processed 950/977 queries...


In [24]:
def normalize_scores(arr, eps: float = 1e-8):
    """
    Min-max normalize scores to [0, 1].
    If all scores are (almost) equal, returns ones.
    """
    arr = np.array(arr, dtype=float)
    if arr.size == 0:
        return arr
    min_v = float(arr.min())
    max_v = float(arr.max())
    if abs(max_v - min_v) < eps:
        return np.ones_like(arr)
    return (arr - min_v) / (max_v - min_v)


In [25]:
from sentence_transformers import CrossEncoder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Load fine-tuned CE for inference
finetuned_ce = CrossEncoder(
    FINETUNED_CE_PATH,
    max_length=512,
    device=DEVICE
)
print("Loaded fine-tuned cross-encoder from:", FINETUNED_CE_PATH)


Loaded fine-tuned cross-encoder from: /content/output/cross_encoder/lepard_finetuned_ce


### Cross-Encoder Reranker

To refine the ranking within the candidate pool, the notebook employs a **cross-encoder reranker** from the `sentence-transformers` library.

Mechanism:

- A cross-encoder jointly encodes a **(query, passage)** pair and produces a single scalar relevance score.
- Unlike a bi-encoder, which encodes queries and passages independently and then combines embeddings (e.g., dot product), the cross-encoder:
  - Concatenates the query and passage into a single input sequence.
  - Lets the transformer model attend across both texts simultaneously.
  - Captures fine-grained interactions such as negation, specific citations, and legal nuances.

At inference time for each query:

1. All candidate passages are paired with the query to form `(query, passage)` pairs.
2. The cross-encoder predicts a relevance score for each pair.
3. Candidates are sorted by these scores.
4. The top-K reranked passages are returned and evaluated using the same retrieval metrics as before.

This reranking step aims to improve the **local ordering** within the candidate pool, especially in cases where the hybrid retriever identifies relevant passages but does not rank them optimally.

In [26]:
def rerank_with_cross_encoder(
    cross_encoder: CrossEncoder,
    candidates,
    top_k: int = TOP_K,
    batch_size: int = 32
):
    """
    Rerank candidate passages using only cross-encoder scores.
    """
    predictions = []
    num_queries = len(candidates)
    print(f"Reranking {num_queries} queries with cross-encoder ONLY (top_k={top_k})...")

    for idx, entry in enumerate(candidates):
        if (idx + 1) % 50 == 0:
            print(f"  Reranked {idx + 1}/{num_queries} queries...")

        query = entry["query"]
        cand_passages = entry["candidate_passages"]

        if not cand_passages:
            predictions.append({"query": query, "retrieved_passages": []})
            continue

        pairs = [[query, passage] for passage in cand_passages]
        scores = cross_encoder.predict(pairs, batch_size=batch_size)
        scores = np.array(scores)

        ranked_idx = np.argsort(-scores)[:top_k]
        reranked_passages = [cand_passages[i] for i in ranked_idx]

        predictions.append({
            "query": query,
            "retrieved_passages": reranked_passages
        })

    return predictions


def evaluate_reranker(
    cross_encoder: CrossEncoder,
    candidates,
    gold_tests,
    top_k: int = TOP_K,
    batch_size: int = 32
):
    ce_predictions = rerank_with_cross_encoder(
        cross_encoder=cross_encoder,
        candidates=candidates,
        top_k=top_k,
        batch_size=batch_size
    )
    results = evaluate_retrieval(
        predictions=ce_predictions,
        gold_standard=gold_tests[:len(ce_predictions)],
        k=top_k
    )
    return ce_predictions, results


In [27]:
def rerank_with_cross_encoder_fused(
    cross_encoder: CrossEncoder,
    candidates,
    top_k: int = TOP_K,
    batch_size: int = 32,
    alpha: float = 0.5
):
    """
    Rerank candidate passages using a fusion of:
      - normalized cross-encoder scores
      - normalized hybrid retrieval scores

    combined_score = alpha * ce_norm + (1 - alpha) * hybrid_norm
    """
    predictions = []
    num_queries = len(candidates)
    print(
        f"Reranking {num_queries} queries with HYBRID + CE fusion "
        f"(alpha={alpha}, top_k={top_k})..."
    )

    for idx, entry in enumerate(candidates):
        if (idx + 1) % 50 == 0:
            print(f"  Reranked {idx + 1}/{num_queries} queries...")

        query = entry["query"]
        cand_passages = entry["candidate_passages"]
        cand_scores = entry.get("candidate_scores", None)

        if not cand_passages:
            predictions.append({"query": query, "retrieved_passages": []})
            continue

        # Cross-encoder scores
        pairs = [[query, passage] for passage in cand_passages]
        ce_scores = np.array(cross_encoder.predict(pairs, batch_size=batch_size))

        # If hybrid scores are missing, fall back to CE-only
        if cand_scores is None:
            combined = ce_scores
        else:
            hybrid_scores = np.array(cand_scores, dtype=float)
            ce_norm = normalize_scores(ce_scores)
            hyb_norm = normalize_scores(hybrid_scores)
            combined = alpha * ce_norm + (1.0 - alpha) * hyb_norm

        ranked_idx = np.argsort(-combined)[:top_k]
        reranked_passages = [cand_passages[i] for i in ranked_idx]

        predictions.append({
            "query": query,
            "retrieved_passages": reranked_passages
        })

    return predictions


def evaluate_reranker_fused(
    cross_encoder: CrossEncoder,
    candidates,
    gold_tests,
    top_k: int = TOP_K,
    batch_size: int = 32,
    alpha: float = 0.5
):
    ce_predictions = rerank_with_cross_encoder_fused(
        cross_encoder=cross_encoder,
        candidates=candidates,
        top_k=top_k,
        batch_size=batch_size,
        alpha=alpha
    )
    results = evaluate_retrieval(
        predictions=ce_predictions,
        gold_standard=gold_tests[:len(ce_predictions)],
        k=top_k
    )
    return ce_predictions, results


In [28]:
# recompute hybrid-only top_k=TOP_K results
hybrid_topk_predictions = []
for test in tests:
    query = test["query"]
    results = hybrid_retriever.retrieve(query, k=TOP_K)
    retrieved = [p for p, s in results]
    hybrid_topk_predictions.append({
        "query": query,
        "retrieved_passages": retrieved
    })

hybrid_results = evaluate_retrieval(
    predictions=hybrid_topk_predictions,
    gold_standard=tests[:len(hybrid_topk_predictions)],
    k=TOP_K
)


In [29]:
# 1) Fine-tuned CE, CE-only reranking
ft_ce_predictions_only, ft_ce_results_only = evaluate_reranker(
    cross_encoder=finetuned_ce,
    candidates=hybrid_candidates,
    gold_tests=tests,
    top_k=TOP_K,
    batch_size=32
)

# 2) Fine-tuned CE + hybrid fused reranking
ALPHA = 0.5
ft_ce_predictions_fused, ft_ce_results_fused = evaluate_reranker_fused(
    cross_encoder=finetuned_ce,
    candidates=hybrid_candidates,
    gold_tests=tests,
    top_k=TOP_K,
    batch_size=32,
    alpha=ALPHA
)

print("=== HYBRID-ONLY vs CE-ONLY vs HYBRID+CE (fused) ===")
for metric in ["exact_match", "span_f1", f"recall@{TOP_K}", f"ndcg@{TOP_K}"]:
    h_val  = hybrid_results.get(metric, float("nan"))
    ce_val = ft_ce_results_only.get(metric, float("nan"))
    fu_val = ft_ce_results_fused.get(metric, float("nan"))
    print(f"{metric:10s}: hybrid={h_val:.4f}  |  CE-only={ce_val:.4f}  |  fused={fu_val:.4f}")

print("num_examples:", hybrid_results.get("num_examples"))


Reranking 977 queries with cross-encoder ONLY (top_k=10)...
  Reranked 50/977 queries...
  Reranked 100/977 queries...
  Reranked 150/977 queries...
  Reranked 200/977 queries...
  Reranked 250/977 queries...
  Reranked 300/977 queries...
  Reranked 350/977 queries...
  Reranked 400/977 queries...
  Reranked 450/977 queries...
  Reranked 500/977 queries...
  Reranked 550/977 queries...
  Reranked 600/977 queries...
  Reranked 650/977 queries...
  Reranked 700/977 queries...
  Reranked 750/977 queries...
  Reranked 800/977 queries...
  Reranked 850/977 queries...
  Reranked 900/977 queries...
  Reranked 950/977 queries...
Reranking 977 queries with HYBRID + CE fusion (alpha=0.5, top_k=10)...
  Reranked 50/977 queries...
  Reranked 100/977 queries...
  Reranked 150/977 queries...
  Reranked 200/977 queries...
  Reranked 250/977 queries...
  Reranked 300/977 queries...
  Reranked 350/977 queries...
  Reranked 400/977 queries...
  Reranked 450/977 queries...
  Reranked 500/977 queries...
 

### Cross-Encoder Reranking on LegalBench-RAG

On LegalBench-RAG, the notebook evaluates three configurations:

1. **Hybrid-only**  
   - Uses the BM25 + Sentence-BERT hybrid scores directly.
2. **Cross-encoder only (CE-only)**  
   - Ignores hybrid scores and ranks candidates purely by cross-encoder scores.
3. **Hybrid + CE fusion**  
   - Combines both scores for each candidate:
     \[
     s_{\text{fused}} = \alpha \cdot s_{\text{CE}} + (1 - \alpha) \cdot s_{\text{hybrid}}
     \]
     with different values of \(\alpha\) (e.g., 0.2, 0.5, 0.8).

Representative results on LegalBench-RAG (for \(\alpha \approx 0.5\)) show:

- **Hybrid-only**:  
  - span_f1 ≈ 0.236  
  - recall@10 ≈ 0.551  
  - nDCG@10 ≈ 0.481
- **CE-only**:  
  - span_f1 ≈ 0.227  
  - recall@10 ≈ 0.514  
  - nDCG@10 ≈ 0.414
- **Hybrid + CE fusion (α ≈ 0.5)**:  
  - span_f1 ≈ 0.239  
  - recall@10 ≈ **0.561**  
  - nDCG@10 ≈ **0.490**

These results indicate:

- On this benchmark, the hybrid retriever alone remains very strong.
- The cross-encoder, especially when fine-tuned on a different legal dataset (LePard), does not outperform the hybrid retriever by itself.
- However, **fusing cross-encoder scores with hybrid scores produces consistent, modest gains**, moving closer to the theoretical candidate upper bound (Recall@10 ≈ 0.5855).

Thus, on LegalBench-RAG the cross-encoder plays a **refinement role**, improving ranking quality within the candidate pool but constrained by both domain shift and the candidate upper bound.


In [30]:
for alpha in [0.2, 0.5, 0.8]:
    _, res = evaluate_reranker_fused(
        cross_encoder=finetuned_ce,
        candidates=hybrid_candidates,
        gold_tests=tests,
        top_k=TOP_K,
        batch_size=32,
        alpha=alpha
    )
    print(f"\nALPHA = {alpha}")
    for metric, val in res.items():
        if metric == "num_examples":
            continue
        print(f"  {metric}: {val:.4f}")


Reranking 977 queries with HYBRID + CE fusion (alpha=0.2, top_k=10)...
  Reranked 50/977 queries...
  Reranked 100/977 queries...
  Reranked 150/977 queries...
  Reranked 200/977 queries...
  Reranked 250/977 queries...
  Reranked 300/977 queries...
  Reranked 350/977 queries...
  Reranked 400/977 queries...
  Reranked 450/977 queries...
  Reranked 500/977 queries...
  Reranked 550/977 queries...
  Reranked 600/977 queries...
  Reranked 650/977 queries...
  Reranked 700/977 queries...
  Reranked 750/977 queries...
  Reranked 800/977 queries...
  Reranked 850/977 queries...
  Reranked 900/977 queries...
  Reranked 950/977 queries...

ALPHA = 0.2
  exact_match: 0.0000
  span_f1: 0.2367
  recall@10: 0.5602
  ndcg@10: 0.4857
Reranking 977 queries with HYBRID + CE fusion (alpha=0.5, top_k=10)...
  Reranked 50/977 queries...
  Reranked 100/977 queries...
  Reranked 150/977 queries...
  Reranked 200/977 queries...
  Reranked 250/977 queries...
  Reranked 300/977 queries...
  Reranked 350/977 

### LePard Dataset and Splitting Strategy

The LePard dataset provides large-scale legal citation data. Each row includes:

- `quote`: the text in a source case that cites another case.
- `destination_context`: the text around the cited destination paragraph.
- Metadata such as `dest_id`, `source_id`, courts, dates, and citations.

For training and evaluation, the notebook:

1. Selects a subset of rows to keep training and evaluation computationally manageable.
2. Splits this subset into **train**, **dev**, and **test** partitions, typically using an 80/10/10 ratio.
3. Optionally performs the split by `dest_id` so that the same destination case does not appear in multiple splits, reducing the risk of target leakage.

Within each split, the notebook treats each `(quote, destination_context)` pair as an example where the destination context should be considered **relevant** to the quote.


In [4]:
# ===== 0. INSTALL & IMPORTS =====
!pip install rank-bm25 sentence-transformers scikit-learn nltk -q

import os
import tarfile
import random
import numpy as np
import pandas as pd
from collections import defaultdict
from typing import List, Dict, Tuple, Optional

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder, InputExample

import torch
from torch.utils.data import DataLoader


In [5]:
import nltk

try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")

try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab")

try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [6]:
# ===== 0.1 NLTK SETUP & BASIC TEXT HELPERS =====
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

def preprocess_text(text: str, lower: bool = True) -> str:
    if not isinstance(text, str):
        text = "" if text is None else str(text)
    if lower:
        text = text.lower()
    text = " ".join(text.split())
    return text.strip()

def tokenize(text: str, remove_stopwords: bool = True) -> List[str]:
    text = preprocess_text(text, lower=True)
    tokens = word_tokenize(text)
    if remove_stopwords:
        tokens = [t for t in tokens if t.isalnum() and t not in stop_words]
    else:
        tokens = [t for t in tokens if t.isalnum()]
    return tokens


In [7]:
# ===== 1. LOAD LEPARD DATASET =====
LEPARD_TAR_PATH = "/content/training_top_100000_data.tar.gz"
EXTRACT_DIR = "/content/lepard_data"

os.makedirs(EXTRACT_DIR, exist_ok=True)

# Extract CSV from tar.gz
with tarfile.open(LEPARD_TAR_PATH, "r:gz") as tar:
    csv_members = [m for m in tar.getmembers() if m.name.endswith(".csv")]
    assert len(csv_members) == 1, f"Expected 1 CSV in tar, found {len(csv_members)}"
    member = csv_members[0]
    tar.extract(member, path=EXTRACT_DIR)
    LEPARD_CSV_PATH = os.path.join(EXTRACT_DIR, member.name)

print("LePard CSV path:", LEPARD_CSV_PATH)

# Load into pandas
df = pd.read_csv(LEPARD_CSV_PATH)

print("Raw LePard shape:", df.shape)
print(df.head())


/tmp/ipython-input-2979966205.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=EXTRACT_DIR)


LePard CSV path: /content/lepard_data/top_100000_data.csv
Raw LePard shape: (100000, 13)
   dest_id  source_id   dest_date  \
0  4140271    3628546  1934-10-17   
1  4140271    3628546  1934-10-17   
2  4140271    3628546  1934-10-17   
3  3836966    5620257  1899-01-09   
4  2184156   12043962  1969-06-17   

                                          dest_court  \
0  United States District Court for the District ...   
1  United States District Court for the District ...   
2  United States District Court for the District ...   
3  United States Court of Appeals for the Third C...   
4  United States Court of Appeals for the Ninth C...   

                                           dest_name  \
0  United States ex rel. Pen Mar Co. v. J. L. Rob...   
1  United States ex rel. Pen Mar Co. v. J. L. Rob...   
2  United States ex rel. Pen Mar Co. v. J. L. Rob...   
3                           Barnes Cycle Co. v. Reed   
4                          Spillman v. United States   

              

In [8]:
# Keep only relevant columns and drop rows with missing text
df = df[["dest_id", "source_id", "passage_id", "quote", "destination_context"]].copy()

df["quote"] = df["quote"].astype(str)
df["destination_context"] = df["destination_context"].astype(str)

df = df.dropna(subset=["quote", "destination_context"])
df = df.reset_index(drop=True)

print("Cleaned LePard shape:", df.shape)


Cleaned LePard shape: (100000, 5)


In [9]:
# ===== 2. BUILD CORPUS =====
# Each passage is identified by (dest_id, passage_id).
# We'll map each unique (dest_id, passage_id, destination_context) to an index.

df["dest_id"] = df["dest_id"].astype(str)
df["passage_id"] = df["passage_id"].astype(str)

unique_passages = df[["dest_id", "passage_id", "destination_context"]].drop_duplicates()

corpus_passages = unique_passages["destination_context"].tolist()
corpus_keys = list(zip(unique_passages["dest_id"], unique_passages["passage_id"]))
docid_to_index = {key: idx for idx, key in enumerate(corpus_keys)}

print("Corpus size (#unique destination_contexts):", len(corpus_passages))


Corpus size (#unique destination_contexts): 99921


In [10]:
from sklearn.model_selection import train_test_split

# ===== 2.0 SUBSAMPLE 5000 ROWS FIRST =====
MAX_ROWS = 10000

if len(df) > MAX_ROWS:
    df_small = df.sample(n=MAX_ROWS, random_state=42).reset_index(drop=True)
    print(f"Subsampled {MAX_ROWS} rows from original {len(df)} rows.")
else:
    df_small = df.reset_index(drop=True)
    print(f"Dataset has only {len(df)} rows; using all rows.")

# ===== 2.1 TRAIN/DEV/TEST SPLIT BY dest_id ON THE SUBSET =====
TRAIN_RATIO = 0.8
DEV_RATIO   = 0.1
TEST_RATIO  = 0.1

dest_ids = df_small["dest_id"].unique()
print("Unique dest_ids in subset:", len(dest_ids))

# Split dest_ids into train / temp
dest_ids_train, dest_ids_tmp = train_test_split(
    dest_ids,
    test_size=(DEV_RATIO + TEST_RATIO),
    random_state=42
)

# Split remaining dest_ids into dev / test
dest_ids_dev, dest_ids_test = train_test_split(
    dest_ids_tmp,
    test_size=TEST_RATIO / (DEV_RATIO + TEST_RATIO),
    random_state=42
)

train_df = df_small[df_small["dest_id"].isin(dest_ids_train)].reset_index(drop=True)
dev_df   = df_small[df_small["dest_id"].isin(dest_ids_dev)].reset_index(drop=True)
test_df  = df_small[df_small["dest_id"].isin(dest_ids_test)].reset_index(drop=True)

print("Train rows:", train_df.shape[0])
print("Dev rows  :", dev_df.shape[0])
print("Test rows :", test_df.shape[0])


Subsampled 10000 rows from original 100000 rows.
Unique dest_ids in subset: 8378
Train rows: 8002
Dev rows  : 1021
Test rows : 977


In [11]:
# ===== 2.2 BUILD TEST QUERIES STRUCTURE =====
# Each test example: query = quote, gold answer = corresponding destination_context.

test_examples = []
for _, row in test_df.iterrows():
    q = row["quote"]
    gold = row["destination_context"]
    test_examples.append({
        "query": q,
        "snippets": [{"answer": gold}]
    })

len(test_examples), test_examples[0]


(977,
 {'query': '[cjertain rulings specifically those which have the ‘effect of changing a practice’-undergo notice-and-comment procedures, 19 C.F.R. § 177.10(e) (1998).',
  'snippets': [{'answer': '” 185 F.3d at 1307.\nMead holds that Haggar’s reach (and thus Chevron deference) does not extend to “ordinary” or “typical” Customs rulings, see 19 C.F.R. § 177.0, 177.1(a) (1998), which do not involve such procedural safeguards as public debate or discussion, are confined to specific facts and parties to a particular transaction at issue, and unlike regulations, are not intended to clarify the rights and obligations of importers beyond the specific matter under review. See 19 C.F.R. However, Mead expressly reserved decision as to whether Chevron deference applies to “'}]})

In [12]:
# ===== 3. EVALUATION FUNCTIONS =====
def exact_match(predicted: str, gold: str) -> bool:
    return preprocess_text(predicted) == preprocess_text(gold)

def span_f1(predicted: str, gold: str) -> float:
    pred_tokens = set(tokenize(predicted, remove_stopwords=False))
    gold_tokens = set(tokenize(gold, remove_stopwords=False))

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    intersection = pred_tokens & gold_tokens
    precision = len(intersection) / len(pred_tokens) if pred_tokens else 0.0
    recall = len(intersection) / len(gold_tokens) if gold_tokens else 0.0

    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def recall_at_k(retrieved_passages: List[str], gold_passages: List[str], k: int = 10) -> float:
    if len(gold_passages) == 0:
        return 1.0 if len(retrieved_passages) == 0 else 0.0

    top_k = retrieved_passages[:k]
    gold_norm = [preprocess_text(g) for g in gold_passages]
    ret_norm  = [preprocess_text(r) for r in top_k]

    matches = 0
    for g in gold_norm:
        for r in ret_norm:
            if g == r or g in r or r in g:
                matches += 1
                break
    return matches / len(gold_passages)

def ndcg_at_k(retrieved_passages: List[str], gold_passages: List[str], k: int = 10) -> float:
    if len(gold_passages) == 0:
        return 1.0 if len(retrieved_passages) == 0 else 0.0

    top_k = retrieved_passages[:k]
    gold_norm = [preprocess_text(g) for g in gold_passages]
    ret_norm  = [preprocess_text(r) for r in top_k]

    relevances = []
    for r in ret_norm:
        relevant = any(g == r or g in r or r in g for g in gold_norm)
        relevances.append(1.0 if relevant else 0.0)

    dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(relevances))
    num_rel = int(min(sum(relevances), k))
    idcg = sum(1.0 / np.log2(i + 2) for i in range(num_rel))
    if idcg == 0:
        return 0.0
    return dcg / idcg

def evaluate_retrieval(predictions: List[Dict], gold_standard: List[Dict], k: int = 10) -> Dict[str, float]:
    assert len(predictions) == len(gold_standard), "Mismatch in #predictions vs #gold"

    exact_matches = []
    span_f1_scores = []
    recall_scores = []
    ndcg_scores = []

    for pred, gold in zip(predictions, gold_standard):
        gold_answers = [s["answer"] for s in gold.get("snippets", [])]
        if not gold_answers:
            continue

        retrieved = pred.get("retrieved_passages", [])
        if not retrieved:
            exact_matches.append(0.0)
            span_f1_scores.append(0.0)
            recall_scores.append(0.0)
            ndcg_scores.append(0.0)
            continue

        top_pred = retrieved[0]
        em = any(exact_match(top_pred, g) for g in gold_answers)
        exact_matches.append(1.0 if em else 0.0)

        best_f1 = max(span_f1(top_pred, g) for g in gold_answers)
        span_f1_scores.append(best_f1)

        rec_k = recall_at_k(retrieved, gold_answers, k=k)
        recall_scores.append(rec_k)

        ndcg_k = ndcg_at_k(retrieved, gold_answers, k=k)
        ndcg_scores.append(ndcg_k)

    return {
        "exact_match": float(np.mean(exact_matches)),
        "span_f1": float(np.mean(span_f1_scores)),
        f"recall@{k}": float(np.mean(recall_scores)),
        f"ndcg@{k}": float(np.mean(ndcg_scores)),
        "num_examples": len(predictions)
    }


In [14]:
# ===== 4. TF-IDF RETRIEVAL =====
TOP_K = 10

# Fit TF-IDF on corpus
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    max_df=0.95,
    min_df=2
)
tfidf_doc_matrix = tfidf_vectorizer.fit_transform(corpus_passages)

def retrieve_tfidf(queries: List[str], top_k: int = 10):
    predictions = []
    for q in queries:
        q_vec = tfidf_vectorizer.transform([q])
        sims = cosine_similarity(q_vec, tfidf_doc_matrix)[0]
        top_idx = np.argsort(-sims)[:top_k]
        retrieved = [corpus_passages[i] for i in top_idx]
        predictions.append({"query": q, "retrieved_passages": retrieved})
    return predictions

test_queries = [ex["query"] for ex in test_examples]
tfidf_predictions = retrieve_tfidf(test_queries, top_k=TOP_K)
tfidf_results = evaluate_retrieval(tfidf_predictions, test_examples, k=TOP_K)

print("=== TF-IDF RESULTS (TEST) ===")
for k, v in tfidf_results.items():
    print(f"{k}: {v:.4f}")


=== TF-IDF RESULTS (TEST) ===
exact_match: 0.0911
span_f1: 0.2586
recall@10: 0.1883
ndcg@10: 0.1406
num_examples: 977.0000


In [15]:
# ===== 5. BM25 RETRIEVAL =====
tokenized_corpus = [tokenize(p, remove_stopwords=True) for p in corpus_passages]
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

def retrieve_bm25(queries: List[str], top_k: int = 10):
    predictions = []
    for q in queries:
        q_tokens = tokenize(q, remove_stopwords=True)
        scores = bm25.get_scores(q_tokens)
        top_idx = np.argsort(-scores)[:top_k]
        retrieved = [corpus_passages[i] for i in top_idx]
        predictions.append({"query": q, "retrieved_passages": retrieved})
    return predictions

bm25_predictions = retrieve_bm25(test_queries, top_k=TOP_K)
bm25_results = evaluate_retrieval(bm25_predictions, test_examples, k=TOP_K)

print("=== BM25 RESULTS (TEST) ===")
for k, v in bm25_results.items():
    print(f"{k}: {v:.4f}")


=== BM25 RESULTS (TEST) ===
exact_match: 0.1198
span_f1: 0.3111
recall@10: 0.2354
ndcg@10: 0.1782
num_examples: 977.0000


In [16]:
# ===== 6. SENTENCE-BERT RETRIEVAL =====
DENSE_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

dense_encoder = SentenceTransformer(DENSE_MODEL_NAME, device=DEVICE)

# Encode corpus
corpus_emb = dense_encoder.encode(
    corpus_passages,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype(np.float32)

def retrieve_dense_sbert(queries: List[str], top_k: int = 10):
    predictions = []
    for q in queries:
        q_emb = dense_encoder.encode([q], convert_to_numpy=True, normalize_embeddings=True)[0]
        scores = corpus_emb @ q_emb  # cosine because normalized
        top_idx = np.argsort(-scores)[:top_k]
        retrieved = [corpus_passages[i] for i in top_idx]
        predictions.append({"query": q, "retrieved_passages": retrieved})
    return predictions

dense_predictions = retrieve_dense_sbert(test_queries, top_k=TOP_K)
dense_results = evaluate_retrieval(dense_predictions, test_examples, k=TOP_K)

print("=== SENTENCE-BERT RESULTS (TEST) ===")
for k, v in dense_results.items():
    print(f"{k}: {v:.4f}")


Batches:   0%|          | 0/1562 [00:00<?, ?it/s]

=== SENTENCE-BERT RESULTS (TEST) ===
exact_match: 0.0727
span_f1: 0.2339
recall@10: 0.1382
ndcg@10: 0.1076
num_examples: 977.0000


In [17]:
# ===== 7. HYBRID BM25 + SBERT =====
def normalize_scores(scores: np.ndarray) -> np.ndarray:
    if scores.size == 0:
        return scores
    min_val = scores.min()
    max_val = scores.max()
    if max_val - min_val < 1e-9:
        return np.ones_like(scores)
    return (scores - min_val) / (max_val - min_val)

def retrieve_hybrid(queries: List[str],
                    top_k: int = 10,
                    bm25_weight: float = 0.55,
                    dense_weight: float = 0.45):
    predictions = []
    for q in queries:
        # BM25 scores
        q_tokens = tokenize(q, remove_stopwords=True)
        bm25_scores = np.array(bm25.get_scores(q_tokens))

        # Dense scores
        q_emb = dense_encoder.encode([q], convert_to_numpy=True, normalize_embeddings=True)[0]
        dense_scores = corpus_emb @ q_emb

        bm25_norm = normalize_scores(bm25_scores)
        dense_norm = normalize_scores(dense_scores)
        combined = bm25_weight * bm25_norm + dense_weight * dense_norm

        top_idx = np.argsort(-combined)[:top_k]
        retrieved = [corpus_passages[i] for i in top_idx]
        predictions.append({"query": q, "retrieved_passages": retrieved})
    return predictions

hybrid_predictions = retrieve_hybrid(test_queries, top_k=TOP_K)
hybrid_results = evaluate_retrieval(hybrid_predictions, test_examples, k=TOP_K)

print("=== HYBRID (BM25 + SBERT) RESULTS (TEST) ===")
for k, v in hybrid_results.items():
    print(f"{k}: {v:.4f}")


=== HYBRID (BM25 + SBERT) RESULTS (TEST) ===
exact_match: 0.1146
span_f1: 0.3035
recall@10: 0.2313
ndcg@10: 0.1732
num_examples: 977.0000


In [18]:
# ===== 7.1 HYBRID CANDIDATES FOR RERANKING =====
CANDIDATE_K = 50  # depth for reranking

hybrid_candidate_predictions = retrieve_hybrid(test_queries, top_k=CANDIDATE_K)
# hybrid_candidate_predictions[i]["retrieved_passages"] is the candidate list.


In [19]:
# ===== 8. CROSS-ENCODER HELPERS =====
def rerank_with_cross_encoder(cross_encoder: CrossEncoder,
                              candidates,
                              top_k: int = 10,
                              batch_size: int = 32):
    """
    candidates: list of dicts
      [{"query": q, "retrieved_passages": [p1, p2, ...]}, ...]
    Returns predictions in same format as evaluate_retrieval() expects.
    """
    predictions = []
    num_queries = len(candidates)
    print(f"Reranking {num_queries} queries with cross-encoder (top_k={top_k})...")

    for idx, entry in enumerate(candidates):
        if (idx + 1) % 50 == 0:
            print(f"  Reranked {idx + 1}/{num_queries} queries...")
        query = entry["query"]
        cand_passages = entry["retrieved_passages"]
        if not cand_passages:
            predictions.append({"query": query, "retrieved_passages": []})
            continue

        pairs = [[query, p] for p in cand_passages]
        scores = np.array(cross_encoder.predict(pairs, batch_size=batch_size))
        top_idx = np.argsort(-scores)[:top_k]
        reranked = [cand_passages[i] for i in top_idx]

        predictions.append({"query": query, "retrieved_passages": reranked})
    return predictions

def evaluate_reranker(cross_encoder: CrossEncoder,
                      candidates,
                      gold_tests,
                      top_k: int = 10,
                      batch_size: int = 32):
    ce_predictions = rerank_with_cross_encoder(
        cross_encoder=cross_encoder,
        candidates=candidates,
        top_k=top_k,
        batch_size=batch_size
    )
    results = evaluate_retrieval(
        predictions=ce_predictions,
        gold_standard=gold_tests[:len(ce_predictions)],
        k=top_k
    )
    return ce_predictions, results


In [20]:
# ===== 8.2 BASELINE CROSS-ENCODER (NO FINE-TUNING) =====
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
FINETUNED_CE_PATH = "/content/output/lepard_finetuned_ce"
os.makedirs(os.path.dirname(FINETUNED_CE_PATH), exist_ok=True)

baseline_ce = CrossEncoder(
    CROSS_ENCODER_MODEL,
    max_length=512,
    device=DEVICE
)

baseline_ce_predictions, baseline_ce_results = evaluate_reranker(
    cross_encoder=baseline_ce,
    candidates=hybrid_candidate_predictions,  # top-50 hybrid candidates
    gold_tests=test_examples,
    top_k=TOP_K,
    batch_size=32
)

print("=== BASELINE CROSS-ENCODER RERANKER RESULTS (TEST) ===")
for k, v in baseline_ce_results.items():
    print(f"{k}: {v:.4f}")


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Reranking 977 queries with cross-encoder (top_k=10)...
  Reranked 50/977 queries...
  Reranked 100/977 queries...
  Reranked 150/977 queries...
  Reranked 200/977 queries...
  Reranked 250/977 queries...
  Reranked 300/977 queries...
  Reranked 350/977 queries...
  Reranked 400/977 queries...
  Reranked 450/977 queries...
  Reranked 500/977 queries...
  Reranked 550/977 queries...
  Reranked 600/977 queries...
  Reranked 650/977 queries...
  Reranked 700/977 queries...
  Reranked 750/977 queries...
  Reranked 800/977 queries...
  Reranked 850/977 queries...
  Reranked 900/977 queries...
  Reranked 950/977 queries...
=== BASELINE CROSS-ENCODER RERANKER RESULTS (TEST) ===
exact_match: 0.1259
span_f1: 0.3186
recall@10: 0.2405
ndcg@10: 0.1847
num_examples: 977.0000


In [21]:
# ===== 9.1 BUILD TRAIN INPUTEXAMPLES FROM LePard TRAIN =====
NUM_NEGATIVES_PER_POS = 2
TRAIN_BATCH_SIZE = 32
NUM_EPOCHS = 2
LEARNING_RATE = 2e-5

# Pre-list of all destination_context texts (global) and indices for neg sampling
all_passages = corpus_passages
num_passages = len(all_passages)

def build_lepard_train_examples(df_split: pd.DataFrame,
                                num_negatives_per_pos: int = 2,
                                seed: int = 42) -> List[InputExample]:
    rng = random.Random(seed)
    examples = []
    for _, row in df_split.iterrows():
        q = row["quote"]
        pos_ctx = row["destination_context"]

        # positive pair
        examples.append(InputExample(
            texts=[q, pos_ctx],
            label=1.0
        ))

        # negatives: same query, random different context
        for _ in range(num_negatives_per_pos):
            while True:
                j = rng.randint(0, num_passages - 1)
                neg_ctx = all_passages[j]
                if neg_ctx != pos_ctx:
                    break
            examples.append(InputExample(
                texts=[q, neg_ctx],
                label=0.0
            ))
    return examples

train_examples = build_lepard_train_examples(
    train_df,
    num_negatives_per_pos=NUM_NEGATIVES_PER_POS,
    seed=42
)
print("Number of train InputExamples (pos+neg):", len(train_examples))

train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=TRAIN_BATCH_SIZE
)


Number of train InputExamples (pos+neg): 24006


In [22]:
# ===== 9.2 DEV PAIRS FOR MONITORING (CLASSIFICATION TASK) =====
def build_pairs_and_labels(df_split: pd.DataFrame,
                           num_negatives_per_pos: int = 2,
                           seed: int = 123):
    rng = random.Random(seed)
    pairs = []
    labels = []

    for _, row in df_split.iterrows():
        q = row["quote"]
        pos_ctx = row["destination_context"]

        # positive
        pairs.append([q, pos_ctx])
        labels.append(1.0)

        # negatives
        for _ in range(num_negatives_per_pos):
            while True:
                j = rng.randint(0, num_passages - 1)
                neg_ctx = all_passages[j]
                if neg_ctx != pos_ctx:
                    break
            pairs.append([q, neg_ctx])
            labels.append(0.0)

    return pairs, labels

dev_pairs, dev_labels = build_pairs_and_labels(
    dev_df,
    num_negatives_per_pos=NUM_NEGATIVES_PER_POS,
    seed=123
)
len(dev_pairs), len(dev_labels)


(3063, 3063)

In [23]:
# ===== 9.3 CEBinaryClassificationEvaluator =====
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator

dev_evaluator = CEBinaryClassificationEvaluator(
    sentence_pairs=dev_pairs,
    labels=dev_labels
)


/tmp/ipython-input-779978920.py:4: DeprecationWarning: This evaluator has been deprecated in favor of the more general CrossEncoderClassificationEvaluator. Please use CrossEncoderClassificationEvaluator instead, which supports both binary and multi-class evaluation. It accepts approximately the same inputs as this evaluator.
  dev_evaluator = CEBinaryClassificationEvaluator(


### Fine-Tuning the Cross-Encoder on LePard

The cross-encoder is fine-tuned on LePard to better model legal citation relevance.

Training setup:

- **Positive examples**  
  - (`quote`, `destination_context`) pairs from the same row, labeled 1.
- **Negative examples**  
  - For each quote, contexts from other destination cases are sampled and paired with the quote, labeled 0.
- **Model**  
  - A cross-encoder from `sentence-transformers` with a single regression output (num_labels=1).
- **Loss and optimization**  
  - Standard regression / binary cross-entropy style loss on the labels.
  - Training is run for several epochs with a small learning rate (e.g., 1e-5).
- **Monitoring**  
  - A `CEBinaryClassificationEvaluator` is used on the dev split to monitor metrics such as:
    - Accuracy
    - F1 score
    - Precision and recall
    - Average precision

Observed training behavior includes:

- Training loss decreasing steadily.
- Dev accuracy and F1 reaching high values, indicating the model learns to distinguish correct citation contexts from negatives.

After training, the fine-tuned cross-encoder is saved and used as a **reranker** in both LePard and LegalBench-RAG retrieval experiments.


In [24]:
# ===== 9.4 FINE-TUNE CROSS-ENCODER ON LePard TRAIN =====
ce_for_training = CrossEncoder(
    CROSS_ENCODER_MODEL,
    num_labels=1,
    max_length=512,
    device=DEVICE
)

warmup_steps = int(len(train_dataloader) * NUM_EPOCHS * 0.1)
print("Warmup steps:", warmup_steps)

ce_for_training.fit(
    train_dataloader=train_dataloader,
    epochs=NUM_EPOCHS,
    warmup_steps=warmup_steps,
    output_path=FINETUNED_CE_PATH,  # best model saved here
    evaluator=dev_evaluator,
    evaluation_steps=len(train_dataloader),  # once per epoch
    optimizer_params={"lr": LEARNING_RATE},
    use_amp=True
)

# Explicit final save
ce_for_training.save(FINETUNED_CE_PATH)
print(f"Fine-tuned model saved to {FINETUNED_CE_PATH}")


Warmup steps: 150


Token indices sequence length is longer than the specified maximum sequence length for this model (541 > 512). Running this sequence through the model will result in indexing errors
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: snavya (snavya-university-of-pennsylvania) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Accuracy,Accuracy Threshold,F1,F1 Threshold,Precision,Recall,Average Precision
751,0.533700,No log,0.853412,-0.523265,0.760849,-1.043290,0.749288,0.772772,0.873650
1502,0.290600,No log,0.857003,0.539409,0.770586,-1.210890,0.700000,0.857003,0.882543


Fine-tuned model saved to /content/output/lepard_finetuned_ce


In [25]:
# ===== 9.5 EVALUATE FINE-TUNED CROSS-ENCODER AS RERANKER =====
finetuned_ce = CrossEncoder(
    FINETUNED_CE_PATH,
    max_length=512,
    device=DEVICE
)

finetuned_ce_predictions, finetuned_ce_results = evaluate_reranker(
    cross_encoder=finetuned_ce,
    candidates=hybrid_candidate_predictions,
    gold_tests=test_examples,
    top_k=TOP_K,
    batch_size=32
)

print("=== FINE-TUNED CROSS-ENCODER RERANKER RESULTS (TEST) ===")
for k, v in finetuned_ce_results.items():
    print(f"{k}: {v:.4f}")


Reranking 977 queries with cross-encoder (top_k=10)...
  Reranked 50/977 queries...
  Reranked 100/977 queries...
  Reranked 150/977 queries...
  Reranked 200/977 queries...
  Reranked 250/977 queries...
  Reranked 300/977 queries...
  Reranked 350/977 queries...
  Reranked 400/977 queries...
  Reranked 450/977 queries...
  Reranked 500/977 queries...
  Reranked 550/977 queries...
  Reranked 600/977 queries...
  Reranked 650/977 queries...
  Reranked 700/977 queries...
  Reranked 750/977 queries...
  Reranked 800/977 queries...
  Reranked 850/977 queries...
  Reranked 900/977 queries...
  Reranked 950/977 queries...
=== FINE-TUNED CROSS-ENCODER RERANKER RESULTS (TEST) ===
exact_match: 0.1198
span_f1: 0.3160
recall@10: 0.2600
ndcg@10: 0.1902
num_examples: 977.0000


In [26]:
print("\n=== SUMMARY COMPARISON ON LePard TEST (TOP_K = {}) ===".format(TOP_K))

def print_result(name, res):
    print(f"\n{name}:")
    for k, v in res.items():
        print(f"  {k}: {v:.4f}")

print_result("TF-IDF", tfidf_results)
print_result("BM25", bm25_results)
print_result("Sentence-BERT", dense_results)
print_result("Hybrid (BM25 + SBERT)", hybrid_results)
print_result("CE Baseline (reranking hybrid candidates)", baseline_ce_results)
print_result("CE Fine-tuned (reranking hybrid candidates)", finetuned_ce_results)

print("\nCOMPARISON (baseline CE vs fine-tuned CE):")
for metric in finetuned_ce_results.keys():
    if metric == "num_examples":
        continue
    base_val = baseline_ce_results.get(metric, float("nan"))
    ft_val   = finetuned_ce_results.get(metric, float("nan"))
    print(f" {metric:10s}: baseline={base_val:.4f}  ->  finetuned={ft_val:.4f}")



=== SUMMARY COMPARISON ON LePard TEST (TOP_K = 10) ===

TF-IDF:
  exact_match: 0.0911
  span_f1: 0.2586
  recall@10: 0.1883
  ndcg@10: 0.1406
  num_examples: 977.0000

BM25:
  exact_match: 0.1198
  span_f1: 0.3111
  recall@10: 0.2354
  ndcg@10: 0.1782
  num_examples: 977.0000

Sentence-BERT:
  exact_match: 0.0727
  span_f1: 0.2339
  recall@10: 0.1382
  ndcg@10: 0.1076
  num_examples: 977.0000

Hybrid (BM25 + SBERT):
  exact_match: 0.1146
  span_f1: 0.3035
  recall@10: 0.2313
  ndcg@10: 0.1732
  num_examples: 977.0000

CE Baseline (reranking hybrid candidates):
  exact_match: 0.1259
  span_f1: 0.3186
  recall@10: 0.2405
  ndcg@10: 0.1847
  num_examples: 977.0000

CE Fine-tuned (reranking hybrid candidates):
  exact_match: 0.1198
  span_f1: 0.3160
  recall@10: 0.2600
  ndcg@10: 0.1902
  num_examples: 977.0000

COMPARISON (baseline CE vs fine-tuned CE):
 exact_match: baseline=0.1259  ->  finetuned=0.1198
 span_f1   : baseline=0.3186  ->  finetuned=0.3160
 recall@10 : baseline=0.2405  -> 

### Retrieval Results on LePard Test Set

On the LePard test split (e.g., ~977 queries, TOP_K = 10), the notebook compares several retrieval configurations:

- **TF-IDF**  
  - Exact match ≈ 0.09  
  - Span F1 ≈ 0.26  
  - Recall@10 ≈ 0.19  
  - nDCG@10 ≈ 0.14

- **BM25**  
  - Exact match ≈ 0.12  
  - Span F1 ≈ 0.31  
  - Recall@10 ≈ 0.24  
  - nDCG@10 ≈ 0.18

- **Sentence-BERT dense retrieval**  
  - Exact match ≈ 0.07  
  - Span F1 ≈ 0.23  
  - Recall@10 ≈ 0.14  
  - nDCG@10 ≈ 0.11

- **Hybrid (BM25 + Sentence-BERT)**  
  - Exact match ≈ 0.11  
  - Span F1 ≈ 0.30  
  - Recall@10 ≈ 0.23  
  - nDCG@10 ≈ 0.17

- **Cross-Encoder baseline (reranking hybrid candidates)**  
  - Exact match ≈ 0.13  
  - Span F1 ≈ 0.32  
  - Recall@10 ≈ 0.24  
  - nDCG@10 ≈ 0.18

- **Cross-Encoder fine-tuned on LePard (reranking hybrid candidates)**  
  - Exact match ≈ 0.12  
  - Span F1 ≈ 0.32  
  - Recall@10 ≈ **0.26**  
  - nDCG@10 ≈ **0.19**

Interpretation:

- **BM25** is a strong baseline on this legal citation task; legal quotes often have high lexical overlap with the relevant destination context.
- **Sentence-BERT alone** underperforms BM25, suggesting that pure semantic similarity does not fully capture the citation patterns in this dataset.
- The **hybrid retriever** is slightly better than BM25 in some metrics but not dramatically so, because lexical signals are already very strong.
- The **cross-encoder baseline** improves over hybrid, showing that joint modeling of `(quote, context)` pairs provides additional signal.
- **Fine-tuning the cross-encoder on LePard** yields a noticeable improvement in Recall@10 and nDCG@10, confirming that in-domain supervision helps the reranker better rank true citation contexts above distractors.


## Discussion and Limitations

Across both LegalBench-RAG and LePard, the combination of a **hybrid BM25 + Sentence-BERT retriever** and a **cross-encoder reranker** delivers consistent but relatively modest improvements over the best unsupervised baselines.

Key factors explaining these observations:

1. **Candidate Pool Upper Bound**  
   - On LegalBench-RAG, the hybrid retriever’s candidate pool has an estimated **Recall@10 upper bound ≈ 0.5855**.
   - The best fused hybrid + cross-encoder configuration reaches Recall@10 ≈ 0.56, leaving very little room for further improvement without changing the first-stage retriever.
   - This indicates that reranking is already using most of the available headroom; further gains require generating **better candidates**, not only better scoring.

2. **Domain Shift Between LePard and LegalBench-RAG**  
   - The cross-encoder is fine-tuned on LePard, a legal citation dataset, whereas LegalBench-RAG is a contract QA dataset with different language patterns and relevance criteria.
   - As a result, the fine-tuned cross-encoder is highly effective on LePard, but only modestly helpful on LegalBench-RAG, where its learned signal does not perfectly align with the benchmark’s notion of relevance.

3. **Strictness of Span-Based Evaluation**  
   - Exact match is close to zero in many experiments because the evaluation uses exact string comparison of spans, while retrieval operates on larger overlapping chunks.
   - Even when the retriever surfaces the correct area of the document, minor differences in snippet boundaries can significantly reduce span F1 and ranking-based scores.

4. **Task Difficulty and Legal Language**  
   - Legal language is dense, formulaic, and highly context-dependent. Multiple passages may look superficially similar, making it difficult for models to sharply distinguish truly relevant spans from near-misses, especially when trained on related but not identical tasks.

Despite these limitations, the experiments demonstrate that:

- **Hybrid lexical + semantic retrieval** is a strong baseline for legal text.
- **Cross-encoder reranking**, particularly when fine-tuned on in-domain supervision (LePard), can reliably improve recall and ranking quality within a candidate pool.
- Understanding and quantifying the **candidate upper bound** is crucial to interpreting retrieval results and deciding whether to invest effort in better retrieval, better reranking, or both.
